# 6.16 Precision-Recall Curve

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.16_precision-recall-curve.ipynb)

Companion code for Section 6.16 of the book. Run the cells in order from top to bottom.

**Example 6.16.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                              roc_auc_score, f1_score)
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

# ── Create imbalanced dataset ─────────────────────────────────────
X, y = make_classification(n_samples=10000, n_features=20, n_informative=10,
                             weights=[0.95, 0.05],  # 5% positive class
                             random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

print(f'Class distribution: {np.bincount(y_te)} (negative/positive)')
print(f'Positive rate: {y_te.mean():.1%}')
print(f'Random PR-AUC baseline: {y_te.mean():.4f}')

# ── Compute PR curves for multiple models ─────────────────────────
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=100, scale_pos_weight=19,
                                              eval_metric='logloss', use_label_encoder=False,
                                              random_state=42, verbosity=0),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Precision-Recall Analysis - Imbalanced Data (5% positive)',
            fontsize=14, fontweight='bold')

ax = axes[0]
baseline_pr = y_te.mean()
ax.axhline(baseline_pr, color='black', ls='--', lw=1.5,
          label=f'Random baseline (P={baseline_pr:.3f})')

print(f'\n{"Model":<22} {"AP (PR-AUC)":>12} {"ROC-AUC":>10} {"F1 (t=0.5)":>12}')
print('='*60)

for name, clf in models.items():
    clf.fit(X_tr_s, y_tr)
    y_prob = clf.predict_proba(X_te_s)[:,1]
    precision, recall, thresholds = precision_recall_curve(y_te, y_prob)
    ap   = average_precision_score(y_te, y_prob)
    rauc = roc_auc_score(y_te, y_prob)
    f1   = f1_score(y_te, (y_prob >= 0.5).astype(int), zero_division=0)
    ax.plot(recall, precision, lw=2.5, label=f'{name} (AP={ap:.4f})')
    print(f'{name:<22} {ap:>12.4f} {rauc:>10.4f} {f1:>12.4f}')

ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision (PPV)', fontsize=12)
ax.set_title('Precision-Recall Curves\n(better = top-right corner)')
ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])

# ── Optimal threshold from PR curve ───────────────────────────────
ax = axes[1]
best_model = list(models.values())[-1]  # XGBoost
y_prob_best = best_model.predict_proba(X_te_s)[:,1]
precision, recall, thresholds = precision_recall_curve(y_te, y_prob_best)
f1_per_thresh = 2*precision[:-1]*recall[:-1]/(precision[:-1]+recall[:-1]+1e-10)
best_idx = np.argmax(f1_per_thresh)
best_t   = thresholds[best_idx]

ax.plot(thresholds, precision[:-1], 'b-', lw=2, label='Precision')
ax.plot(thresholds, recall[:-1],    'r-', lw=2, label='Recall')
ax.plot(thresholds, f1_per_thresh,  'g-', lw=2, label='F1')
ax.axvline(best_t, color='black', ls='--', lw=2,
          label=f'Best threshold={best_t:.3f}\nF1={f1_per_thresh[best_idx]:.4f}')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('XGBoost: Precision/Recall/F1 vs Threshold\n(Optimal threshold for imbalanced data)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Apply optimal threshold
y_pred_opt = (y_prob_best >= best_t).astype(int)
print(f'\nOptimal threshold: {best_t:.4f}')
print(f'Precision at optimal: {precision[best_idx]:.4f}')
print(f'Recall at optimal:    {recall[best_idx]:.4f}')
print(f'F1 at optimal:        {f1_per_thresh[best_idx]:.4f}')